# Snake-RepairLLaMA — Trained Adapter Evaluation (Run 1)

**Goal:** measure how the **trained Snake-RepairLLaMA LoRA adapter** (on top of `codellama/CodeLlama-7b-hf`) performs on the QuixBugs (40 bugs) and BugsInPy (161 verified bugs) IR4/OR2 evaluation sets — using the **same prompts, sampling protocol, and metrics** as the baseline run, so results are directly comparable.

**Runtime:** Designed for **Colab Free T4** (16 GB VRAM). Loads the base model in **8-bit** + the LoRA adapter (~30 MB) on top. Same code also runs on a RunPod 4090 / A10 in FP16 — see the BugsInPy cell for the alternative settings.

**Inference protocol (matches the RepairLLaMA paper, identical to baseline):**
- 10 candidate patches per bug
- Sampling: `do_sample=True, temperature=1.0, top_p=0.95`  (NOT beam search)
- `max_new_tokens=256`
- 8-bit base model + LoRA adapter attached via `PeftModel.from_pretrained`

**Cell-by-cell flow:**
1. Setup: clone repo, install deps
2. HF login (token must have access to BOTH the base model and your adapter repo if it's gated)
3. Sanity check: load base + adapter, inspect tokenized prompt, generate on 1 bug
4. Run full inference on QuixBugs (~40 bugs × 10 = 400 generations)
5. Run full inference on BugsInPy (~161 bugs × 10 = 1610 generations)
6. Score & print Top-1 / Top-3 / Top-10 exact / AST / compile / buried-fix
7. (Optional) Save results JSONL to Google Drive

**STOP-AND-CHECK before step 4:** the sanity-check generation in step 3 should look like a sensible Python completion of the buggy line. If it's garbage, the adapter's training prompt format almost certainly doesn't match this eval format (`<FILL_ME>`-style FIM). Don't burn the next 30 min of compute — go check the training prompts first.

## 1. Setup — clone repo & install deps

In [1]:
import os


In [2]:
!pwd

/workspace/snake-repairllama-baseline


In [5]:
!rm -rf /content/snake-repairllama-baseline

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/AliSuleman27/snake-repairllama-baseline.git"
# REPO_DIR = "/workspace/snake-repairllama-baseline"
REPO_DIR = "d:/snake-repairllama-baseline"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("cwd:", os.getcwd())
print("contents:", os.listdir(REPO_DIR))

cwd: d:\snake-repairllama-baseline
contents: ['.cfg', '.git', '.gitignore', '.venv', 'codellama models comparision.png', 'context.md', 'data', 'notebooks', 'other model benchmarking results.png', 'README.md', 'requirements.txt', 'results', 'scripts', 'src', 'train']


In [2]:
# Install deps. Colab has torch + transformers preinstalled; we add the rest.
%pip install "transformers>=4.40" "peft>=0.10" "accelerate>=0.30" "bitsandbytes>=0.43" "datasets>=2.18" tqdm


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. HF login

CodeLlama is gated. Add your HF token as a Colab Secret named `HF_TOKEN`, **or** paste it in the prompt below.

In [3]:
import os

token = None
# try:
#     from google.colab import userdata
#     token = userdata.get("HF_TOKEN")
# except Exception:
#     pass

if not token:
    from getpass import getpass
    token = getpass("Paste your HF token (write or read access): ")

from huggingface_hub import login
login(token=token, add_to_git_credential=False)
print("HF login OK")

Paste your HF token (write or read access):  ········


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful
HF login OK


In [4]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

## 3. Sanity check — load model & test on 1 bug

Run this BEFORE the full eval. Verifies model loads, prompt is well-formed, and a generation looks like a Python snippet (not garbage). ~3 min on T4.

In [5]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [2]:
# Base model — must match what the adapter was trained on top of.
MODEL_NAME    = "codellama/CodeLlama-7b-hf"
ADAPTER_NAME  = "alisuleman525/snake-repairllama-7b-fim-run3"
RUN_TAG       = "snakellama_run3"

assert ADAPTER_NAME != "FILL_ME_IN", "Set ADAPTER_NAME to your trained adapter (HF repo or local path) before running."
print("MODEL_NAME  :", MODEL_NAME)
print("ADAPTER_NAME:", ADAPTER_NAME)
print("RUN_TAG     :", RUN_TAG)

MODEL_NAME  : codellama/CodeLlama-7b-hf
ADAPTER_NAME: alisuleman525/snake-repairllama-7b-fim-run3
RUN_TAG     : snakellama_run3


In [8]:
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# No quantization on A100 80GB — fp16 fits and avoids the peft+bnb 8-bit bug
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

print(f"Attaching adapter: {ADAPTER_NAME}")
model = PeftModel.from_pretrained(base, ADAPTER_NAME)
model.eval()
print("Model loaded. VRAM:", torch.cuda.memory_allocated() / 1e9, "GB")
print("Adapter config:", model.peft_config)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Attaching adapter: alisuleman525/snake-repairllama-7b-fim-run3


adapter_model.safetensors:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

Model loaded. VRAM: 20.601095168 GB
Adapter config: {'default': LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='codellama/CodeLlama-7b-hf', revision=None, task_type='CAUSAL_LM', inference_mode=True, r=8, target_modules={'v_proj', 'q_proj'}, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))}


In [9]:
with open("data/quixbugs_eval.jsonl") as f:
    bug = json.loads(f.readline())

print("=== INPUT (IR4) ===")
print(bug["input"])
print("=== GOLD (OR2) ===")
print(bug["output"])

=== INPUT (IR4) ===
def bitcount(n):
    count = 0
    while n:
# Buggy code:
#         n ^= n - 1
<FILL_ME>
        count += 1
    return count

=== GOLD (OR2) ===
        n &= n - 1



In [11]:
inputs = tokenizer(bug["input"], return_tensors="pt")
print("ids:", inputs.input_ids)
print("max id:", inputs.input_ids.max().item(), "vocab:", model.config.vocab_size)
print("decoded:", tokenizer.decode(inputs.input_ids[0]))

ids: tensor([[    1, 32007,   822,  2586,  2798, 29898, 29876,  1125,    13,  1678,
          2302,   353, 29871, 29900,    13,  1678,  1550,   302, 29901,    13,
         29937, 28209,  1927,   775, 29901,    13, 29937,   308,   302,  6228,
         29922,   302,   448, 29871, 29896,    13, 32008,    13,  4706,  2302,
          4619, 29871, 29896,    13,  1678,   736,  2302,    13, 32009]])
max id: 32009 vocab: 32016
decoded: <s> <PRE> def bitcount(n):
    count = 0
    while n:
# Buggy code:
#         n ^= n - 1
 <SUF>
        count += 1
    return count
 <MID>


In [7]:
# Pick the first QuixBugs bug, generate 1 patch, eyeball the output.

inputs = tokenizer(bug["input"], return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=256,
        do_sample=True, temperature=1.0, top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
raw = tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== RAW GENERATION ===")
print(raw)

from src.postprocess import extract_patch
print("=== EXTRACTED (strict) ===")
print(extract_patch(raw, mode="strict"))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


=== RAW GENERATION ===
# Better code:
        if n & 1:
            count += 1
#        if n % 2 == 1:
#            count += 1
        
        n >>= 1
        if n % 2 == 1:
=== EXTRACTED (strict) ===
# Better code:
        if n & 1:
            count += 1
#        if n % 2 == 1:
#            count += 1
        
        n >>= 1
        if n % 2 == 1:


## 4. Full inference — QuixBugs (40 bugs × 10 samples)

~30 min on T4. Resumes if interrupted (re-run the cell).

In [13]:
# Free the sanity-check model first to make room (we re-load via the helper)
import gc
try:
    del model, base, tokenizer
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

from src.inference import run_inference

run_inference(
    eval_jsonl="data/quixbugs_eval.jsonl",
    output_jsonl=f"results/quixbugs_{RUN_TAG}.jsonl",
    model_name=MODEL_NAME,
    adapter_name=ADAPTER_NAME,        # <-- attaches the trained LoRA on top
    n_samples=10,
    load_in_8bit=False,
)

[inference] Loaded 40 bugs from data/quixbugs_eval.jsonl
[inference] Loading tokenizer: codellama/CodeLlama-7b-hf
[inference] Loading base model: codellama/CodeLlama-7b-hf


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[inference] Attaching adapter: alisuleman525/snake-repairllama-7b-fim-run3


bugs:   0%|          | 0/40 [00:00<?, ?it/s]

[inference] Done in 107.9s (2.7s/bug)
[inference] Wrote results/quixbugs_snakellama_run3.jsonl


In [16]:
import gc
# del model, tokenizer
gc.collect(); torch.cuda.empty_cache()


## 5. Full inference — BugsInPy (196 bugs × 10 samples)

~3-4 hours on T4 free. If your Colab session might disconnect, run with `limit=50` first to get partial numbers, or use the resume-on-rerun behavior.

In [17]:
from src.inference import run_inference

# === Colab T4 (8-bit) — default. Long BugsInPy inputs need sub_batch_size=2
# to avoid VRAM OOM. ~3-5 hours on T4.
# run_inference(
#     eval_jsonl="data/bugsinpy_eval_verified.jsonl",
#     output_jsonl=f"results/bugsinpy_{RUN_TAG}.jsonl",
#     model_name=MODEL_NAME,
#     adapter_name=ADAPTER_NAME,
#     n_samples=10,
#     load_in_8bit=False,
#     sub_batch_size=2,
# )

# === RunPod 4090 / FP16 alternative — uncomment if running on RunPod.
# Should finish in ~15-30 min. Comment out the T4 call above first.
run_inference(
    eval_jsonl="data/bugsinpy_eval_verified.jsonl",
    output_jsonl=f"results/bugsinpy_{RUN_TAG}.jsonl",
    model_name=MODEL_NAME,
    adapter_name=ADAPTER_NAME,
    n_samples=10,
    load_in_8bit=False,
    sub_batch_size=10,
)

[inference] Loaded 161 bugs from data/bugsinpy_eval_verified.jsonl
[inference] Resuming: 3 bugs already done
[inference] Loading tokenizer: codellama/CodeLlama-7b-hf
[inference] Loading base model: codellama/CodeLlama-7b-hf


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[inference] Attaching adapter: alisuleman525/snake-repairllama-7b-fim-run3


bugs:   0%|          | 0/158 [00:00<?, ?it/s]

[inference] Done in 1307.9s (8.3s/bug)
[inference] Wrote results/bugsinpy_snakellama_run3.jsonl


## 6. Score & report

Top-1 / Top-3 / Top-10 for **exact-match**, **AST-match**, **compile**, and **buried-fix** (gold appears anywhere in the generation — useful to see if the base model 'knows' the fix even if it can't isolate it cleanly).

In [3]:
from src.metrics import evaluate_file, print_report

qb = evaluate_file(
    inference_jsonl=f"results/quixbugs_{RUN_TAG}.jsonl",
    eval_jsonl="data/quixbugs_eval.jsonl",
)
print_report(f"QuixBugs — Snake-RepairLLaMA trained adapter ({RUN_TAG})", qb)


  QuixBugs — Snake-RepairLLaMA trained adapter (snakellama_run3)
  Total bugs: 40    Bugs with plausibility data: 0
  Top-1  Exact     :   19 / 40 ( 47.5%)
  Top-1  AST       :   21 / 40 ( 52.5%)
  Top-1  Compile   :   25 / 40 ( 62.5%)

  Top-3  Exact     :   28 / 40 ( 70.0%)
  Top-3  AST       :   30 / 40 ( 75.0%)
  Top-3  Compile   :   26 / 40 ( 65.0%)
  Top-3  Buried    :   28 / 40 ( 70.0%)

  Top-10 Exact     :   32 / 40 ( 80.0%)
  Top-10 AST       :   33 / 40 ( 82.5%)
  Top-10 Compile   :   29 / 40 ( 72.5%)
  Top-10 Buried    :   32 / 40 ( 80.0%)


In [4]:
from src.metrics import evaluate_file, print_report

bp = evaluate_file(
    inference_jsonl=f"results/bugsinpy_{RUN_TAG}.jsonl",
    eval_jsonl="data/bugsinpy_eval_verified.jsonl",
)
print_report(f"BugsInPy — Snake-RepairLLaMA trained adapter ({RUN_TAG})", bp)


  BugsInPy — Snake-RepairLLaMA trained adapter (snakellama_run3)
  Total bugs: 161    Bugs with plausibility data: 0
  Top-1  Exact     :   11 / 161 (  6.8%)
  Top-1  AST       :   12 / 161 (  7.5%)
  Top-1  Compile   :   99 / 161 ( 61.5%)

  Top-3  Exact     :   14 / 161 (  8.7%)
  Top-3  AST       :   15 / 161 (  9.3%)
  Top-3  Compile   :  108 / 161 ( 67.1%)
  Top-3  Buried    :   17 / 161 ( 10.6%)

  Top-10 Exact     :   23 / 161 ( 14.3%)
  Top-10 AST       :   25 / 161 ( 15.5%)
  Top-10 Compile   :  116 / 161 ( 72.0%)
  Top-10 Buried    :   28 / 161 ( 17.4%)


<unknown>:2: SyntaxWarning: invalid escape sequence '\s'
<unknown>:2: SyntaxWarning: invalid escape sequence '\s'


## 7. (Optional) Save results to Google Drive

Persists `results/*.jsonl` so you can compare against the trained-adapter run later.